In [70]:
!pip install opencv-python

In [71]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

In [72]:
def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';

      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;

      await video.play();

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;

      canvas.getContext('2d').drawImage(video, 0, 0);

      stream.getVideoTracks()[0].stop();
      div.remove();

      return canvas.toDataURL('image/jpeg', quality);
    }
  ''')

  display(js)

  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])

  with open(filename, 'wb') as f:
    f.write(binary)

  return filename

In [ ]:
filename = take_photo()
print("Image saved as:", filename)

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

In [ ]:
img = cv2.imread(filename)

gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

faces = face_cascade.detectMultiScale(gray, 1.3, 5)

for (x, y, w, h) in faces:
    cv2.rectangle(img,(x,y),(x+w,y+h),(0,255,0),2)

cv2_imshow(img)

In [ ]:
registered = take_photo("registered.jpg")
print("Registered face saved")

In [ ]:
login_img = take_photo("login.jpg")
print("Login image captured")

In [ ]:
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

img1 = cv2.imread("registered.jpg",0)
img2 = cv2.imread("login.jpg",0)

# resize images
img1 = cv2.resize(img1,(200,200))
img2 = cv2.resize(img2,(200,200))

# calculate difference
difference = cv2.absdiff(img1,img2)
score = np.sum(difference)

print("Difference Score:", score)

if score < 1000000:
    print("Login Successful ✅")
else:
    print("Access Denied ❌")